# EarningsSignal Agent — 运行过程可视化
## 五层 Agent Pipeline 交互式回放

> **回放模式**：本 Notebook 不依赖向量数据库、BGE-M3 模型或 GPU 资源。所有数据均来自 `agent_core/feature_history.jsonl`（160KB）——这是 Agent 31 轮完整运行的审计追踪文件，记录了每轮特征的假设定义、提取指令、验证指标和诊断修复结果。
>
> **运行方式**：执行 `pip install matplotlib ipywidgets` → Run All Cells → 下拉菜单选择特征
>
> **数据来源**：`feature_history.jsonl`（仓库内附带）| 无需联网 | 无需 API Key | 无需 GPU

---

### ⚠️ 关于实时运行

本 Demo 展示的是 Agent 的**运行过程回放**（基于已持久化的审计轨迹），而非实时调用 LLM。完整的实时运行需要：
- **FAISS 向量索引**（~4GB）：需运行 `phase0_pipeline/build_transcript_index.py` + `build_theory_index.py` 构建
- **BGE-M3 模型权重**（~2.2GB）：需下载至 `model/` 目录
- **财报转录文本数据集**（~500MB）：需从 HuggingFace 下载
- **LLM API Key**：需配置 `.env` 中的 `SILICONFLOW_API_KEY`

以上文件因体积过大无法上传至 GitHub，详见 `phase0_pipeline/README.md` 中的构建说明。

---

基于 31 轮真实 Agent 运行数据，逐步展示每个特征的完整处理链路：
**假设生成 → 批量提取 → 统计验证 → 规则治理 → 诊断修复**

使用下拉菜单选择任意特征，观察五层 Agent 每一步的输入、输出和决策。

In [ ]:
import json, textwrap, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Arc
from matplotlib.patches import Rectangle as MplRectangle
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets

warnings.filterwarnings('ignore')

matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

# ── 配色体系 ──
C = {
    'h_bg': '#1a5276', 'e_bg': '#1e8449', 'v_bg': '#b9770e',
    'g_bg': '#922b21', 'd_bg': '#6c3483',
    'pass_green': '#27ae60', 'fail_red': '#c0392b',
    'accent_blue': '#2980b9', 'light_gray': '#ecf0f1',
    'text_dark': '#2c3e50', 'text_mid': '#7f8c8d',
    'white': '#ffffff', 'bg': '#f8f9fa',
}

print('✓ 环境就绪')

In [ ]:
# ── 加载数据 ──
HISTORY_PATH = Path('agent_core/feature_history.jsonl')

features = []
with open(HISTORY_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        d = json.loads(line)
        if 'feature_spec' in d and 'governance_result' in d:
            features.append(d)

# 按名称去重（保留最后版本）
seen = {}
for f in features:
    name = f['governance_result']['feature_name']
    seen[name] = f
features_unique = list(seen.values())

# 构建下拉选项：PASS 排前面
features_sorted = sorted(features_unique,
    key=lambda f: (not f['governance_result']['passed'], -f['governance_result']['ic']))

feature_options = []
for f in features_sorted:
    g = f['governance_result']
    status = '✓' if g['passed'] else '✗'
    label = f"{status} {g['feature_name']}  (IC={g['ic']:+.4f}, t={g['t_stat']:+.2f})"
    feature_options.append((label, g['feature_name']))

# 修复谱系
repair_lineages = {}
for f in features_unique:
    name = f['governance_result']['feature_name']
    base = name.replace('_v2_v2', '').replace('_v2', '')
    if base != name:
        repair_lineages.setdefault(base, []).append(name)

print(f'✓ 加载 {len(features_unique)} 个不重复特征')
print(f'  PASS: {sum(1 for f in features_unique if f["governance_result"]["passed"])} 个')
print(f'  修复谱系: {len(repair_lineages)} 个 (含 v2/v3)')

In [ ]:
# ── 辅助函数 ──
def wrap(text, width=50):
    """中文文本换行"""
    if not text:
        return ''
    lines = []
    cur = ''
    for ch in text:
        cur += ch
        if len(cur) >= width:
            lines.append(cur)
            cur = ''
    if cur:
        lines.append(cur)
    return '\n'.join(lines)

def draw_agent_card(ax, x, y, w, h, title, content_lines, color, status=None):
    """绘制单个 Agent 卡片"""
    # 背景卡片
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                          facecolor=color, edgecolor='white', linewidth=2, alpha=0.95, zorder=2)
    ax.add_patch(box)

    # 标题
    ax.text(x + w/2, y + h - 0.35, title, ha='center', va='top',
            fontsize=10, fontweight='bold', color='white')

    # 状态标签
    if status:
        status_color = C['pass_green'] if 'PASS' in status else C['fail_red']
        ax.text(x + w - 0.2, y + h - 0.35, status, ha='right', va='top',
                fontsize=8, fontweight='bold', color=status_color,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.95))

    # 内容行
    for i, line in enumerate(content_lines):
        if line.startswith('##'):  # 小标题
            ax.text(x + 0.3, y + h - 0.8 - i*0.45, line.replace('##', ''),
                    ha='left', va='top', fontsize=7.5, fontweight='bold', color='white', alpha=0.95)
        else:
            ax.text(x + 0.3, y + h - 0.8 - i*0.45, line,
                    ha='left', va='top', fontsize=7, color='white', alpha=0.85)

def draw_arrow(ax, x1, y1, x2, y2, color='#2c3e50', lw=2):
    """绘制连接箭头"""
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw,
                               connectionstyle='arc3,rad=0'), zorder=1)

def draw_gate_row(ax, x, y, w, gates_status):
    """绘制四道门控的横向状态条"""
    gate_names = ['G1 |IC|>0.015', 'G2 zr<45%', 'G3 |t|>1.5', 'G4 dir>60%']
    gw = w / 4
    for i, (name, (passed_gate, detail)) in enumerate(zip(gate_names, gates_status)):
        gx = x + i * gw
        color = C['pass_green'] if passed_gate else C['fail_red']
        icon = '✓' if passed_gate else '✗'
        box = FancyBboxPatch((gx+0.05, y), gw-0.1, 0.65, boxstyle='round,pad=0.05',
                              facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.85, zorder=2)
        ax.add_patch(box)
        ax.text(gx + gw/2, y + 0.43, f'{icon} {name}', ha='center', va='center',
                fontsize=7.5, fontweight='bold', color='white')
        ax.text(gx + gw/2, y + 0.1, detail, ha='center', va='center',
                fontsize=6.5, color='white', alpha=0.9)

print('✓ 绘图函数就绪')

In [ ]:
# ── 架构总览图 ──
def draw_architecture_overview():
    fig, ax = plt.subplots(1, 1, figsize=(20, 5))
    ax.set_xlim(0, 24)
    ax.set_ylim(0, 6)
    ax.axis('off')
    fig.patch.set_facecolor(C['bg'])

    agents = [
        ('Hypothesis\nAgent', 'RAG理论约束\nLLM假设生成', C['h_bg'], 2),
        ('Extraction\nAgent', 'GPU向量检索\nLLM批量打分', C['e_bg'], 7),
        ('Validation\nAgent', 'LightGBM\nOOS验证', C['v_bg'], 12),
        ('Governance\nAgent', '四指标硬规则\nPASS/FAIL', C['g_bg'], 17),
    ]

    for name, desc, color, x in agents:
        draw_agent_card(ax, x, 2, 3.5, 3, name, [f'##{desc}'], color)

    # 箭头
    for i in range(3):
        draw_arrow(ax, agents[i][3]+3.5, 3.5, agents[i+1][3], 3.5)

    # G→D FAIL 分支
    draw_agent_card(ax, 17, 0.3, 3.5, 1.2, 'DiagnosisAgent',
                    ['##RAG根因诊断 + 8种失败模式'], C['d_bg'])
    ax.annotate('FAIL →', xy=(17, 2), xytext=(18.75, 1.5),
                arrowprops=dict(arrowstyle='->', color=C['fail_red'], lw=2.5),
                fontsize=8, color=C['fail_red'], fontweight='bold')
    ax.annotate('repair_spec →\n(反馈至Hypothesis)', xy=(5, 3.5), xytext=(15, 1),
                arrowprops=dict(arrowstyle='->', color=C['d_bg'], lw=2,
                               connectionstyle='arc3,rad=-0.5', linestyle='dashed'),
                fontsize=7.5, color=C['d_bg'], fontweight='bold', ha='center')

    # RAG 知识库
    rag = FancyBboxPatch((0.5, 0.3), 4, 1.2, boxstyle='round,pad=0.08',
                          facecolor=C['light_gray'], edgecolor='#bdc3c7', linewidth=1.5, zorder=1)
    ax.add_patch(rag)
    ax.text(2.5, 1.2, '📚 RAG 知识库', ha='center', va='center', fontsize=9, fontweight='bold', color=C['text_dark'])
    ax.text(2.5, 0.6, '20 Papers · 6 理论簇 · BGE-M3', ha='center', va='center', fontsize=7.5, color=C['text_mid'])
    ax.plot([3, 3.75], [1.5, 2], 'k--', lw=0.8, alpha=0.3)
    ax.plot([18.75, 18.75], [1.5, 2], 'k--', lw=0.8, alpha=0.3)

    ax.set_title('EarningsSignal Agent — 五层自主发掘架构', fontsize=14, fontweight='bold',
                 color=C['text_dark'], pad=15)
    plt.tight_layout()
    plt.show()

draw_architecture_overview()

---
## 交互式特征探查

从下拉菜单选择一个特征，逐层观察 Agent 的输入、输出和决策。

In [ ]:
# ── 核心渲染函数：单特征完整流水线 ──
def render_feature_pipeline(feature_name):
    """渲染选中特征的完整五层 Agent 流水线"""

    # 查找特征数据
    feature = seen.get(feature_name)
    if feature is None:
        print(f'特征 {feature_name} 未找到')
        return

    spec = feature['feature_spec']
    gov = feature['governance_result']
    is_pass = gov['passed']
    diagnosis = gov.get('diagnosis', {})

    # ── 创建画布 ──
    n_rows = 6 if not is_pass else 5
    fig = plt.figure(figsize=(22, 4.2 * n_rows))
    fig.patch.set_facecolor(C['bg'])

    # 总标题
    status_text = '✅ PASS' if is_pass else '❌ FAIL'
    status_color = C['pass_green'] if is_pass else C['fail_red']
    fig.suptitle(f'{status_text}  {feature_name}    '
                 f'IC={gov["ic"]:+.4f}  t={gov["t_stat"]:+.2f}  '
                 f'zero_ratio={gov["zero_ratio"]:.1%}  dir_consistency={gov.get("direction_consistency",0):.0%}',
                 fontsize=13, fontweight='bold', color=status_color, y=0.995)

    gs = fig.add_gridspec(n_rows, 2, height_ratios=[1]*n_rows,
                          hspace=0.3, wspace=0.02,
                          left=0.05, right=0.95, top=0.93, bottom=0.03)

    row = 0

    # ═══════ Layer 1: HypothesisAgent ═══════
    ax = fig.add_subplot(gs[row, :])
    ax.set_xlim(0, 20); ax.set_ylim(0, 5); ax.axis('off')

    # 卡片
    card = FancyBboxPatch((0.3, 0.3), 19.4, 4.4, boxstyle='round,pad=0.1',
                           facecolor=C['h_bg'], edgecolor='white', linewidth=3, alpha=0.95)
    ax.add_patch(card)
    ax.text(10, 4.5, '① HypothesisAgent — 理论约束假设生成', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')

    # 定义
    ax.text(0.8, 3.8, f'定义: {spec.get("definition", "N/A")}',
            fontsize=9, color='white', fontweight='bold')

    # 理论来源
    theory = spec.get('theory_basis', {})
    if theory:
        ax.text(0.8, 3.0, f'📖 理论来源: {theory.get("source","")}', fontsize=8, color='white', alpha=0.9)
        ax.text(0.8, 2.5, f'   {wrap(theory.get("implication",""), 90)}', fontsize=7.5, color='white', alpha=0.8)
    else:
        # 种子特征无 theory_basis，显示 RAG refs
        rags = spec.get('_theory_rag_refs', [])
        if rags:
            rag_str = ' | '.join([f'{r["paper"][:50]}...' for r in rags[:3]])
            ax.text(0.8, 3.0, f'📖 RAG 检索 (top-{len(rags)}):', fontsize=8, color='white', alpha=0.9)
            ax.text(0.8, 2.5, f'   {rag_str}', fontsize=7, color='white', alpha=0.75)

    # 提取指令
    instr = spec.get('extraction_instruction', '')
    ax.text(0.8, 1.8, f'📝 提取指令 ({len(instr)} 字符):', fontsize=8, color='white', alpha=0.9)
    ax.text(0.8, 1.2, wrap(instr, 95), fontsize=7, color='white', alpha=0.75)

    # 检索query & 条件域
    ax.text(11, 3.8, f'🔍 检索 query: {spec.get("retrieval_query","")}',
            fontsize=8, color='white', alpha=0.85)
    scope = spec.get('condition_scope', {})
    ax.text(11, 3.2, f'🎯 条件域: section={scope.get("section_type",[])}, '
            f'speaker={scope.get("speaker_role",[])}, sector={scope.get("sector","all")}',
            fontsize=7.5, color='white', alpha=0.8)
    ax.text(11, 2.7, f'📊 score_range={spec.get("score_range",[])}, top_k={spec.get("top_k","")}',
            fontsize=7.5, color='white', alpha=0.8)
    cluster = spec.get('_theory_cluster', 'seed')
    ax.text(11, 2.2, f'🏷 理论簇: {cluster}', fontsize=8, color='white', alpha=0.85)

    row += 1

    # ═══════ Layer 2: ExtractionAgent ═══════
    ax = fig.add_subplot(gs[row, :])
    ax.set_xlim(0, 20); ax.set_ylim(0, 3.5); ax.axis('off')

    card = FancyBboxPatch((0.3, 0.3), 19.4, 3.0, boxstyle='round,pad=0.1',
                           facecolor=C['e_bg'], edgecolor='white', linewidth=3, alpha=0.95)
    ax.add_patch(card)
    ax.text(10, 3.1, '② ExtractionAgent — GPU 检索 + LLM 批量打分', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')

    # Score 分布横向条形图
    score_dist = gov.get('score_dist', {})
    if score_dist:
        sd_sorted = sorted(score_dist.items(), key=lambda x: float(x[0]))
        labels_sd = [f'得分 {k}' for k, v in sd_sorted]
        values_sd = [v*100 for k, v in sd_sorted]

        ax_bar = ax.inset_axes([0.05, 0.3, 0.55, 2.2])
        colors_bar = ['#e74c3c' if float(k) < 0 else '#bdc3c7' if float(k) == 0 else '#2ecc71'
                      for k, v in sd_sorted]
        ax_bar.barh(range(len(labels_sd)), values_sd, color=colors_bar, edgecolor='white', height=0.6)
        ax_bar.set_yticks(range(len(labels_sd)))
        ax_bar.set_yticklabels(labels_sd, fontsize=8, color='white')
        ax_bar.set_xlabel('%', fontsize=8, color='white')
        ax_bar.set_facecolor('none')
        ax_bar.tick_params(colors='white', labelsize=7)
        for spine in ax_bar.spines.values():
            spine.set_color('white')
        for i, v in enumerate(values_sd):
            ax_bar.text(v+1, i, f'{v:.1f}%', va='center', fontsize=7.5, color='white', fontweight='bold')

    # 右侧统计
    ax.text(13, 2.5, f'📦 检索: 95万chunk GPU matmul → Top-3000', fontsize=8, color='white', alpha=0.85)
    ax.text(13, 2.0, f'📋 批量: 20 eps/batch × 20 线程并发', fontsize=8, color='white', alpha=0.85)
    ax.text(13, 1.5, f'⏱ 耗时: 全量 11,363 eps ~4-5min', fontsize=8, color='white', alpha=0.85)
    ax.text(13, 1.0, f'🎯 零值 (=0): {gov["zero_ratio"]:.1%}', fontsize=9, color='white', fontweight='bold')

    row += 1

    # ═══════ Layer 3: ValidationAgent ═══════
    ax = fig.add_subplot(gs[row, :])
    ax.set_xlim(0, 20); ax.set_ylim(0, 4); ax.axis('off')

    card = FancyBboxPatch((0.3, 0.3), 19.4, 3.5, boxstyle='round,pad=0.1',
                           facecolor=C['v_bg'], edgecolor='white', linewidth=3, alpha=0.95)
    ax.add_patch(card)
    ax.text(10, 3.7, '③ ValidationAgent — LightGBM Walk-Forward OOS 验证', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')

    # 四大指标卡片
    metrics = [
        ('IC', f'{gov["ic"]:+.4f}', 1.5),
        ('NW t-stat', f'{gov["t_stat"]:+.3f}', 6),
        ('Zero Ratio', f'{gov["zero_ratio"]:.1%}', 10.5),
        ('Dir Consistency', f'{gov.get("direction_consistency",0):.0%}', 15),
    ]
    for label, value, mx in metrics:
        mbox = FancyBboxPatch((mx-1, 0.5), 3.5, 2.5, boxstyle='round,pad=0.08',
                               facecolor='white', edgecolor='white', linewidth=1, alpha=0.2)
        ax.add_patch(mbox)
        ax.text(mx+0.75, 2.3, label, ha='center', va='center', fontsize=8, color='white', alpha=0.8)
        ax.text(mx+0.75, 1.4, value, ha='center', va='center', fontsize=18, fontweight='bold', color='white')

    row += 1

    # ═══════ Layer 4: GovernanceAgent ═══════
    ax = fig.add_subplot(gs[row, :])
    ax.set_xlim(0, 20); ax.set_ylim(0, 3); ax.axis('off')

    card = FancyBboxPatch((0.3, 0.3), 19.4, 2.5, boxstyle='round,pad=0.1',
                           facecolor=C['g_bg'], edgecolor='white', linewidth=3, alpha=0.95)
    ax.add_patch(card)
    ax.text(10, 2.6, '④ GovernanceAgent — 四指标硬规则门控', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')

    # 门控状态
    failures = gov.get('failures', [])
    gate_checks = [
        (abs(gov['ic']) > 0.015, f'|IC|={abs(gov["ic"]):.4f}'),
        (gov['zero_ratio'] < 0.45, f'zr={gov["zero_ratio"]:.1%}'),
        (abs(gov['t_stat']) > 1.5, f'|t|={abs(gov["t_stat"]):.2f}'),
        (gov.get('direction_consistency', 0) > 0.60, f'dir={gov.get("direction_consistency",0):.0%}'),
    ]
    draw_gate_row(ax, 0.5, 0.4, 19, gate_checks)

    # PASS/FAIL 大标签
    verdict_text = 'PASS ✓' if is_pass else 'FAIL ✗'
    verdict_color = C['pass_green'] if is_pass else C['fail_red']
    ax.text(18.5, 1.8, verdict_text, ha='center', va='center',
            fontsize=22, fontweight='bold', color=verdict_color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=verdict_color, linewidth=2))

    if failures:
        ax.text(0.8, 1.6, f'拦截原因: {" | ".join(failures)}',
                fontsize=7.5, color='white', alpha=0.85)

    row += 1

    # ═══════ Layer 5: DiagnosisAgent (仅FAIL) ═══════
    if not is_pass and diagnosis:
        ax = fig.add_subplot(gs[row, :])
        ax.set_xlim(0, 20); ax.set_ylim(0, 4.5); ax.axis('off')

        card = FancyBboxPatch((0.3, 0.3), 19.4, 4.0, boxstyle='round,pad=0.1',
                               facecolor=C['d_bg'], edgecolor='white', linewidth=3, alpha=0.95)
        ax.add_patch(card)
        ax.text(10, 4.1, '⑤ DiagnosisAgent — RAG 根因诊断 + 修复方案', ha='center', va='center',
                fontsize=12, fontweight='bold', color='white')

        ax.text(0.8, 3.3, f'🔍 根因: {wrap(diagnosis.get("root_cause",""), 100)}',
                fontsize=8, color='white', fontweight='bold')
        ax.text(0.8, 2.2, f'🔧 修复: {wrap(diagnosis.get("fix",""), 100)}',
                fontsize=8, color='white', alpha=0.9)
        ax.text(0.8, 1.2, f'⛔ 规避: {wrap(diagnosis.get("avoid",""), 100)}',
                fontsize=8, color='white', alpha=0.85)

        # RAG refs
        rags = diagnosis.get('rag_refs', [])
        if rags:
            ax.text(11, 3.3, '📚 文献证据:', fontsize=7.5, color='white', fontweight='bold')
            for j, ref in enumerate(rags[:4]):
                ax.text(11, 2.8-j*0.4,
                        f'  [{j+1}] {ref["paper"][:55]}... (p.{ref["page"]}, sim={ref["score"]:.3f})',
                        fontsize=6.5, color='white', alpha=0.8)
            queries = diagnosis.get('queries_used', [])
            if queries:
                ax.text(11, 0.8, f'🔍 检索query: {queries[0][:80]}...',
                        fontsize=6.5, color='white', alpha=0.75)

        row += 1

    # ═══════ 行业/年份零值面板 (所有特征) ═══════
    ax_sector = fig.add_subplot(gs[row, 0])
    ax_sector.set_xlim(0, 10); ax_sector.set_ylim(0, 5); ax_sector.axis('off')

    zero_by_sector = gov.get('zero_by_sector', {})
    if zero_by_sector:
        sectors = list(zero_by_sector.keys())
        zr_s = [zero_by_sector[s]*100 for s in sectors]
        colors_zs_actual = []
        for v in zr_s:
            if v > 50:
                colors_zs_actual.append(C['fail_red'])
            elif v > 35:
                colors_zs_actual.append('#f39c12')
            else:
                colors_zs_actual.append(C['pass_green'])

        ax_bar_s = ax_sector.inset_axes([0.15, 0.2, 0.8, 4.3])
        ax_bar_s.barh(range(len(sectors)), zr_s, color=colors_zs_actual, edgecolor='white', height=0.6)
        ax_bar_s.set_yticks(range(len(sectors)))
        ax_bar_s.set_yticklabels(sectors, fontsize=7)
        ax_bar_s.axvline(x=45, color=C['fail_red'], linestyle='--', lw=1.5)
        ax_bar_s.set_xlabel('Zero Ratio %', fontsize=8)
        ax_bar_s.set_title('按行业零值率', fontsize=9, fontweight='bold')
        for i, v in enumerate(zr_s):
            ax_bar_s.text(v+0.5, i, f'{v:.1f}%', va='center', fontsize=6.5)
        ax_bar_s.set_xlim(0, max(zr_s)*1.3)

    # 年份零值率
    ax_year = fig.add_subplot(gs[row, 1])
    ax_year.set_xlim(0, 10); ax_year.set_ylim(0, 5); ax_year.axis('off')

    zero_by_year = gov.get('zero_by_year', {})
    if zero_by_year:
        years_y = list(zero_by_year.keys())
        zr_y = [zero_by_year[y]*100 for y in years_y]

        ax_py = ax_year.inset_axes([0.15, 0.2, 0.8, 4.3])
        ax_py.plot(years_y, zr_y, 'o-', color=C['accent_blue'], lw=2, markersize=6,
                   markerfacecolor='white', markeredgewidth=1.5)
        ax_py.axhline(y=45, color=C['fail_red'], linestyle='--', lw=1.5)
        ax_py.fill_between(years_y, zr_y, alpha=0.1, color=C['accent_blue'])
        ax_py.set_xlabel('Year', fontsize=8)
        ax_py.set_title('按年份零值率趋势', fontsize=9, fontweight='bold')
        ax_py.tick_params(labelsize=7)

    plt.show()

print('✓ 渲染函数就绪')

In [ ]:
# ── 修复谱系对比视图 ──
def render_repair_lineage(base_name):
    """并排对比修复谱系中所有版本"""
    versions = [base_name]
    v2 = base_name + '_v2'
    v3 = base_name + '_v2_v2'
    if v2 in seen:
        versions.append(v2)
    if v3 in seen:
        versions.append(v3)

    if len(versions) < 2:
        print(f'{base_name} 无修复谱系（仅 v1）')
        return

    n = len(versions)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 9))
    if n == 1:
        axes = [axes]
    fig.patch.set_facecolor(C['bg'])

    for idx, (vname, ax) in enumerate(zip(versions, axes)):
        f = seen[vname]
        g = f['governance_result']
        spec = f['feature_spec']
        diag = g.get('diagnosis', {})
        is_pass = g['passed']
        is_last = (idx == n-1)

        ax.set_xlim(0, 10); ax.set_ylim(0, 14); ax.axis('off')

        # 卡片背景
        bg_color = C['pass_green'] if is_pass else C['fail_red']
        alpha = 0.95 if is_last else 0.7
        card = FancyBboxPatch((0.3, 0.3), 9.4, 13.4, boxstyle='round,pad=0.1',
                               facecolor=bg_color, edgecolor='white', linewidth=3, alpha=alpha)
        ax.add_patch(card)

        # 版本标签
        gen_label = ['v1 (原始)', 'v2 (一次修复)', 'v3 (二次修复)'][idx]
        ax.text(5, 13.2, f'{gen_label}', ha='center', fontsize=13, fontweight='bold', color='white')
        status = '✓ PASS' if is_pass else '✗ FAIL'
        ax.text(5, 12.5, f'{vname}  {status}', ha='center', fontsize=9,
                color='white', fontfamily='monospace', alpha=0.9)

        # 指标
        y = 11.5
        ax.text(1, y, f'IC = {g["ic"]:+.4f}', fontsize=11, fontweight='bold', color='white')
        ax.text(1, y-0.6, f't-stat = {g["t_stat"]:+.3f}', fontsize=9, color='white')
        ax.text(1, y-1.2, f'zero_ratio = {g["zero_ratio"]:.1%}', fontsize=9, color='white')
        ax.text(1, y-1.8, f'dir_consistency = {g.get("direction_consistency",0):.0%}', fontsize=9, color='white')
        ax.text(1, y-2.5, f'指令长度 = {len(spec.get("extraction_instruction",""))} 字符',
                fontsize=9, color='white')

        # Score分布
        sd = g.get('score_dist', {})
        if sd:
            ax.text(1, y-3.3, 'Score 分布:', fontsize=8, color='white', fontweight='bold')
            sd_sorted = sorted(sd.items(), key=lambda x: float(x[0]))
            sd_str = '  |  '.join([f'{k}: {v:.1%}' for k, v in sd_sorted])
            ax.text(1, y-3.8, sd_str, fontsize=7, color='white', alpha=0.85)

        # 诊断
        if diag and not is_pass:
            ax.text(1, 5.5, '🩺 Diagnosis:', fontsize=9, color='white', fontweight='bold')
            ax.text(1, 4.8, f'根因: {wrap(diag.get("root_cause",""), 55)}', fontsize=7, color='white', alpha=0.85)
            ax.text(1, 3.5, f'修复: {wrap(diag.get("fix",""), 55)}', fontsize=7, color='white', alpha=0.85)

        # 失败项
        failures = g.get('failures', [])
        if failures:
            ax.text(1, 1.5, f'拦截: {" | ".join(failures)}', fontsize=7, color='white', alpha=0.8)

        # 版本间箭头
        if idx < n-1:
            ax.annotate('diagnosis\n+\nrepair →', xy=(10.5, 7), xytext=(9.5, 7),
                        fontsize=8, color='#7f8c8d', ha='center', annotation_clip=False)

    fig.suptitle(f'修复谱系: {base_name} ({n} 代迭代)', fontsize=14, fontweight='bold',
                 color=C['text_dark'], y=1.01)
    plt.tight_layout()
    plt.show()

# 找出所有修复谱系
lineage_bases = set()
for name in seen:
    if '_v2' in name:
        base = name.replace('_v2_v2', '').replace('_v2', '')
        lineage_bases.add(base)

print(f'✓ 修复谱系渲染函数就绪 ({len(lineage_bases)} 个谱系)')
for lb in sorted(lineage_bases):
    vs = [lb] + ([lb+'_v2'] if lb+'_v2' in seen else []) + ([lb+'_v2_v2'] if lb+'_v2_v2' in seen else [])
    print(f'  {lb}: {" → ".join(vs)}')

In [ ]:
# ── 融合评估总结面板 ──
def render_fusion_summary():
    """多因子融合 vs M0 基线评估"""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.patch.set_facecolor(C['bg'])

    methods = ['M0 基线', '融合(13文本)', '精选(7正IC)']
    colors_bar = [C['text_mid'], C['accent_blue'], C['d_bg']]

    # ICIR
    ax = axes[0]
    vals = [1.108, 1.277, 1.209]
    bars = ax.bar(methods, vals, color=colors_bar, edgecolor='white', width=0.35)
    for b, v, prev in zip(bars, vals, [None, vals[0], vals[0]]):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{v:.3f}',
                ha='center', fontsize=12, fontweight='bold')
        if prev and v > prev:
            ax.annotate(f'+{(v-prev)/prev*100:.1f}%', xy=(b.get_x()+b.get_width()/2, v),
                        xytext=(b.get_x()+b.get_width()/2+0.1, v+0.07),
                        fontsize=14, ha='center', color=C['pass_green'], fontweight='bold')
    ax.set_title('ICIR ↑', fontsize=14, fontweight='bold', color=C['text_dark'])
    ax.grid(True, alpha=0.2, axis='y')

    # IR
    ax = axes[1]
    vals = [2.484, 2.733, 2.326]
    bars = ax.bar(methods, vals, color=colors_bar, edgecolor='white', width=0.35)
    for b, v, prev in zip(bars, vals, [None, vals[0], vals[0]]):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{v:.3f}',
                ha='center', fontsize=12, fontweight='bold')
        if prev and v > prev:
            ax.annotate(f'+{(v-prev)/prev*100:.1f}%', xy=(b.get_x()+b.get_width()/2, v),
                        xytext=(b.get_x()+b.get_width()/2+0.1, v+0.07),
                        fontsize=14, ha='center', color=C['pass_green'], fontweight='bold')
    ax.set_title('IR ↑', fontsize=14, fontweight='bold', color=C['text_dark'])
    ax.grid(True, alpha=0.2, axis='y')

    # MDD
    ax = axes[2]
    vals = [-1.37, -0.49, -1.62]
    bars = ax.bar(methods, vals, color=colors_bar, edgecolor='white', width=0.35)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v-0.08, f'{v:+.2f}%',
                ha='center', fontsize=12, fontweight='bold', color='white')
    ax.annotate('回撤↓64%', xy=(1, -0.49), xytext=(1.3, -0.25),
                fontsize=11, color=C['pass_green'], fontweight='bold')
    ax.set_title('Max Drawdown ↓', fontsize=14, fontweight='bold', color=C['text_dark'])
    ax.grid(True, alpha=0.2, axis='y')

    # NW t-stat
    ax = axes[3]
    vals = [5.075, 5.674, 4.888]
    bars = ax.bar(methods, vals, color=colors_bar, edgecolor='white', width=0.35)
    for b, v, prev in zip(bars, vals, [None, vals[0], vals[0]]):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05, f'{v:.3f}',
                ha='center', fontsize=12, fontweight='bold')
        if prev and v > prev:
            ax.annotate(f'+{(v-prev)/prev*100:.1f}%', xy=(b.get_x()+b.get_width()/2, v),
                        xytext=(b.get_x()+b.get_width()/2+0.1, v+0.15),
                        fontsize=14, ha='center', color=C['pass_green'], fontweight='bold')
    ax.set_title('NW t-stat ↑', fontsize=14, fontweight='bold', color=C['text_dark'])
    ax.grid(True, alpha=0.2, axis='y')

    fig.suptitle('多因子融合评估 (S&P 500, 2021-2023 OOS)', fontsize=15,
                 fontweight='bold', color=C['text_dark'], y=1.03)
    plt.tight_layout()
    plt.show()

    # 打印汇总表
    print(f"{'方法':<30} {'IC':>8} {'ICIR':>8} {'NW_t':>8} {'IR':>8} {'MDD':>8}")
    print('-'*70)
    print(f"{'M0 (BASE13 基线)':<30} {0.114:>+8.4f} {1.108:>8.3f} {5.075:>8.3f} {2.484:>8.3f} {'-1.37%':>8}")
    print(f"{'融合 (13 文本特征)':<30} {0.114:>+8.4f} {1.277:>8.3f} {5.674:>8.3f} {2.733:>8.3f} {'-0.49%':>8}")
    print(f"{'精选 (7 正IC特征)':<30} {0.117:>+8.4f} {1.209:>8.3f} {4.888:>8.3f} {2.326:>8.3f} {'-1.62%':>8}")

render_fusion_summary()

In [ ]:
# ── 交互式控件 ──
feature_dropdown = widgets.Dropdown(
    options=[(label, name) for label, name in feature_options],
    value=feature_options[0][1],
    description='选择特征:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)

# 修复谱系下拉
lineage_options = [('— 选择修复谱系 —', '')]
for lb in sorted(lineage_bases):
    lineage_options.append((f'📜 {lb}', lb))

lineage_dropdown = widgets.Dropdown(
    options=lineage_options,
    value='',
    description='修复谱系:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)

output = widgets.Output()

def on_feature_change(change):
    with output:
        clear_output(wait=True)
        render_feature_pipeline(change['new'])

def on_lineage_change(change):
    if change['new']:
        with output:
            clear_output(wait=True)
            render_repair_lineage(change['new'])

feature_dropdown.observe(on_feature_change, names='value')
lineage_dropdown.observe(on_lineage_change, names='value')

ui = widgets.VBox([
    widgets.HBox([widgets.Label('🔍 单特征流水线回放:', layout=widgets.Layout(width='140px')),
                 feature_dropdown]),
    widgets.HBox([widgets.Label('📜 修复谱系对比:', layout=widgets.Layout(width='140px')),
                 lineage_dropdown]),
    output
])

display(ui)

# 默认渲染第一个特征
with output:
    render_feature_pipeline(feature_options[0][1])